# Stream D - Final Analysis and ρ Curve

This notebook computes the recovery ratio ρ(d) based on logs generated during Phase 1 (contamination) and Phase 2 (recovery).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/NLP_project/src')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from metrics import calculate_rho, bootstrap_ci

In [ ]:
doses = [0.01, 0.05, 0.10, 0.25]
seeds = [0, 1, 2]

def load_logs(stage, dose, seed):
    path = f'/content/drive/MyDrive/NLP_project/results/{stage}_logs/dose_{dose}_seed_{seed}.csv'
    if os.path.exists(path):
        return pd.read_csv(path)
    return None

def find_crossing_step(df, threshold, condition='greater'):
    """
    Find the first step where toxicity crosses the threshold.
    condition: 'greater' for contamination, 'less' for recovery
    """
    if df is None:
        return float('inf')
    if condition == 'greater':
        crossed = df[df['toxicity'] > threshold]
    else:
        crossed = df[df['toxicity'] <= threshold]
        
    if len(crossed) == 0:
        return float('inf')
    return crossed.iloc[0]['step']

In [ ]:
# Attempt to load baseline toxicity T0 and sigma from a clean run (dose=0.0)
baseline_df = load_logs('contamination', dose=0.0, seed=0)
if baseline_df is not None and not baseline_df.empty:
    # Compute from the dedicated clean run
    T0 = baseline_df['toxicity'].mean()
    sigma = baseline_df['toxicity'].std() if len(baseline_df) > 1 else 0.02
    print("Loaded T0 and sigma from baseline run.")
else:
    print("Baseline run not found. Using placeholders. Make sure to run dose=0.0 to generate baselines.")
    T0 = 0.15
    sigma = 0.02

tau = T0 + 3 * sigma
epsilon = 0.5 * sigma

print(f"Thresholds: T0={T0:.4f}, tau={tau:.4f}, epsilon={epsilon:.4f}")

In [ ]:
results = []

for dose in doses:
    rhos = []
    for seed in seeds:
        contam_df = load_logs('contamination', dose, seed)
        recov_df = load_logs('recovery', dose, seed)
        
        steps_to_corrupt = find_crossing_step(contam_df, tau, condition='greater')
        steps_to_recover = find_crossing_step(recov_df, T0 + epsilon, condition='less')
        
        if steps_to_corrupt != float('inf') and steps_to_recover != float('inf'):
            rho = calculate_rho(steps_to_corrupt, steps_to_recover)
            rhos.append(rho)
            results.append({
                'dose': dose, 'seed': seed, 
                'steps_to_corrupt': steps_to_corrupt, 
                'steps_to_recover': steps_to_recover, 
                'rho': rho
            })
        else:
            print(f"Run dose={dose} seed={seed} did not fully cross thresholds.")
            
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Calculate aggregations & CI
agg_results = []
for dose in doses:
    dose_data = results_df[results_df['dose'] == dose]['rho'].values
    if len(dose_data) > 0:
        mean_rho, lower, upper = bootstrap_ci(dose_data)
        agg_results.append({
            'dose': dose, 
            'mean_rho': mean_rho, 
            'lower_ci': lower, 
            'upper_ci': upper,
            'count': len(dose_data)
        })
        
        # 1-sided t-test: is rho > 1?
        if len(dose_data) > 1:
            _, p_val = stats.ttest_1samp(dose_data, popmean=1.0, alternative='greater')
            print(f"Dose {dose}: rho > 1 p-value = {p_val:.4f}")
            
agg_df = pd.DataFrame(agg_results)
agg_df

In [ ]:
# Plot ρ Curve
plt.figure(figsize=(8, 5))
plt.errorbar(agg_df['dose'], agg_df['mean_rho'], 
             yerr=[agg_df['mean_rho'] - agg_df['lower_ci'], agg_df['upper_ci'] - agg_df['mean_rho']], 
             fmt='-o', capsize=5, label=r'$\rho(d)$')
plt.axhline(y=1.0, color='r', linestyle='--', label='Symmetry (ρ=1)')
plt.xlabel('Contamination Dose (d)')
plt.ylabel(r'Recovery Ratio $\rho(d)$')
plt.title('Toxicity Asymmetry: Recovery Ratio vs Contamination Dose')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()